### Import thư viện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import keras
import os
import numpy as np
import cv2
import random
from tensorflow.keras.models import load_model
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from PIL import Image

# CHUẨN BỊ VÀ TIỀN XỬ LÝ DỮ LIỆU

### Đọc dữ liệu ảnh

In [ ]:
# Chuyển ảnh về ma trận ảnh Gray và resize về kích thước 224x224
def matrix_images(image_folder, image_files):
    matrix_img = []
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        # Resize ảnh về kích thước 224x224
        image = cv2.resize(image, (224, 224))
        matrix_img.append(image)

    return matrix_img

### Chuẩn hóa dữ liệu

In [ ]:
# ------------------------------
# Chuẩn hóa dữ liệu
# ------------------------------
def scaling_img(data_img):
    # Chuyển danh sách các mảng numpy thành một mảng numpy đa chiều
    data_array = np.array(data_img, dtype=np.float32)

    # Chuẩn hóa dữ liệu
    scaled_img = data_array / 255.0
    return scaled_img

In [ ]:
# Giảm nhiễu cho ảnh radar bằng cách sử dụng bộ lọc Gaussian
def calculate_sigma(kernel_size):
    sigma = (kernel_size - 1) / 6.0
    return sigma

def reduce_noise(image, kernel_size=3):
    sigma = calculate_sigma(kernel_size)
    smoothed_image = cv2.GaussianBlur(image, (kernel_size, kernel_size), sigma)
    return smoothed_image

# TẠO DỮ LIỆU ĐẦU VÀO

### Tạo nhãn cho dữ liệu

In [ ]:
# ------------------------------
# Tạo nhãn chuỗi dữ liệu các ảnh liên tiếp từ dữ liệu hình ảnh
# ------------------------------
def create_image_sequences(data, time_steps):
    num_samples, height, width, channels = data.shape
    num_frames = num_samples - time_steps
    input_sequences = np.zeros((num_frames, time_steps, height, width, channels), dtype=np.float32)
    labels = np.zeros((num_frames, height, width, channels), dtype=np.float32)

    for i in range(num_frames):
        input_sequences[i] = data[i:i+time_steps]
        labels[i] = data[i+time_steps]

    return input_sequences, labels

In [ ]:
def create_seq2img(data, time_steps_input=3, time_steps_output=6):
    N, C, H, W = data.shape
    M = N - time_steps_input - time_steps_output + 1
    if M <= 0:
        raise ValueError("Không đủ khung hình để tạo chuỗi")
    X = np.stack([data[i : i + time_steps_input] for i in range(M)])
    Y = np.stack([data[i + time_steps_input : i + time_steps_input + time_steps_output] for i in range(M)])
    return X.astype(np.float32), Y.astype(np.float32)

### Chia dữ liệu

In [ ]:
# ------------------------------
# Chia dữ liệu
# ------------------------------
def Split_Data(X_data, y_data):
    # Chia dữ liệu thành các tập train, val, test
    size = int(len(X_data) * 0.8)
    size_val = int((len(X_data) - size) / 2)

    X_train = X_data[:size]
    X_val = X_data[size:size + size_val]
    X_test = X_data[size + size_val:]

    y_train = y_data[:size]
    y_val = y_data[size:size + size_val]
    y_test = y_data[size + size_val:]

    return X_train, y_train, X_val, y_val, X_test, y_test

### Chuẩn bị đầu vào

In [ ]:
# Đọc từng file ảnh và sắp xếp theo thứ tự tên tệp
image_folder = '/content/drive/MyDrive/BacSon/processed_data_img'
image_files = sorted([f for f in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, f))],
                     key=lambda x: int(''.join(filter(str.isdigit, x))) if any(c.isdigit() for c in x) else x)

In [ ]:
# Đọc dữ liệu ảnh
data_img = matrix_images(image_folder, image_files[-64:])

timesteps = 4

if len(data_img) == 0:
    raise ValueError("Không có ảnh nào được xử lý, vui lòng kiểm tra lại thư mục!")

# Thực hiện chuẩn hóa và khử nhiễu dữ liệu
data_img_processing = []
data_img_scaled = scaling_img(data_img)
for img in data_img_scaled:
    data_noise_img = reduce_noise(img, kernel_size=3)
    data_img_processing.append(data_noise_img)

# Chuyển thành numpy array và thêm trục kênh
data_image = np.expand_dims(np.array(data_img_processing), axis=-1)

# Tạo chuỗi dữ liệu ảnh liên tiếp
datas, labels = create_image_sequences(data_image, timesteps)

seq_data, seq_label = create_seq2img(data_image, time_steps_input=timesteps, time_steps_output=6)

# Chia dữ liệu
X_train, y_train, X_val, y_val, X_test, y_test = Split_Data(datas, labels)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print()
_, _, _, _, X_test_seq, y_test_seq = Split_Data(seq_data, seq_label)
print("X_test_seq shape:", X_test_seq.shape)
print("y_test_seq shape:", y_test_seq.shape)

# Xây dựng mô hình CNN + GRU

In [ ]:
def build_vgg16_gru_model(input_shape=(4, 224, 224, 1)):
    input_layer = layers.Input(shape=input_shape)

    # Sử dụng TimeDistributed để áp dụng các lớp của VGG16 cho từng khung thời gian
    x = layers.TimeDistributed(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))(input_layer)
    x = layers.TimeDistributed(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2), strides=(2, 2)))(x)

    # Block 2
    x = layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2), strides=(2, 2)))(x)

    # Block 3
    x = layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2), strides=(2, 2)))(x)

    # Block 4
    x = layers.TimeDistributed(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2), strides=(2, 2)))(x)

    # Block 5
    x = layers.TimeDistributed(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2), strides=(2, 2)))(x)

    # Flatten đầu ra từ VGG16
    x = layers.TimeDistributed(layers.Flatten())(x)

    # GRU cho sequence modeling
    x = layers.GRU(256, activation='tanh', return_sequences=True)(x)
    x = layers.Dropout(0.5)(x)
    x = layers.GRU(256, activation='tanh', return_sequences=False)(x)

    # Fully connected layers
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer
    x = layers.Dense((224 * 224), activation='sigmoid')(x)
    output_layer = layers.Reshape((224, 224, 1))(x)

    # Định nghĩa mô hình
    model = models.Model(inputs=input_layer, outputs=output_layer)

    return model

### Đánh giá và test mô hình

In [ ]:
def plot_loss_model(history):
    # Kiểm tra loại của history để trích xuất giá trị loss phù hợp
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']

    # Vẽ biểu đồ
    plt.figure(figsize=(8, 5))
    plt.plot(train_loss, linestyle='-', label='Train loss')
    plt.plot(val_loss, linestyle='-', label='Val loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train loss và Val loss của mô hình')
    plt.yscale('log')  # Scale y-axis to logarithmic scale
    plt.legend()  # Thêm nhãn chú thích
    plt.show()

In [ ]:
# Dự đoán ảnh mới
def prediction(model, X_test):
    predicted = model.predict(X_test)
    predicted_images = np.array(predicted)

    return predicted_images

In [ ]:
# Đánh giá kết quả dự báo
def evaluate_test(y_test, predicted_images):
    predicted_images = np.array(predicted_images)
    y_test = np.array(y_test)

    # Đánh giá MSE MAE R^2
    mse = mean_squared_error(y_test.flatten(), predicted_images.flatten())
    print("MSE =", mse)
    mae = mean_absolute_error(y_test.flatten(), predicted_images.flatten())
    print("MAE =", mae)
    rmse = np.sqrt(mse)
    print("RMSE =", rmse)
    rmae = np.sqrt(mae)
    print("RMAE =", rmae)
    r2 = r2_score(y_test.flatten(), predicted_images.flatten())
    print("R2 =", r2)

In [ ]:
# Vẽ các ảnh dự đoán
def plots(predicted_images):
    # Hiển thị các ảnh dự đoán
    fig, axs = plt.subplots(1, len(predicted_images), figsize=(15, 25))
    for i in range(len(predicted_images)):
        axs[i].imshow(predicted_images[i])  # Hiển thị ảnh cuối cùng trong mỗi mẫu dự đoán
        axs[i].set_title(f"Ảnh dự đoán {i+1}")
    plt.show()

# Áp dụng GA tối ưu Adam

In [ ]:
# Định nghĩa hàm fitness_function
model = build_vgg16_gru_model()

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='mean_squared_error')
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Dừng sớm
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=1, mode='min', restore_best_weights=True)

In [ ]:
# Lưu model tốt nhất
model_path = '/content/drive/MyDrive/BacSon/EAI_2024/KetQua/CNN_GRU_v2.keras'
# os.makedirs(os.path.dirname(model_path), exist_ok=True)
model_checkpoint = ModelCheckpoint(model_path, monitor='val_loss', mode='min', save_best_only=True, verbose=1)

# Training
history = model.fit(X_train, y_train,
                    epochs=200,
                    batch_size=4,
                    validation_data=(X_val, y_val),
                    callbacks=[model_checkpoint, early_stopping])

In [ ]:
plot_loss_model(history)

In [ ]:
# Dự đoán trên model tốt nhất
# model = load_model('/content/drive/MyDrive/BacSon/EAI_2024/KetQua/best_model_CNN_GRU.keras')
predicted_images_UNet = prediction(model, X_test)

# Đánh giá kết quả dự đoán
evaluate_test(y_test, predicted_images_UNet)

# Hiển thị ảnh dự đoán-
plots(predicted_images_UNet)

In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim

def evaluate_image_metrics(y_test, predictions, value_range=(0.0, 1.0)):
    mse_list = []
    mae_list = []
    rmse_list = []
    rmae_list = []
    ssim_list = []

    num_samples = y_test.shape[0]
    min_val, max_val = value_range
    data_range = max_val - min_val if (max_val - min_val) > 0 else 1.0

    for i in range(num_samples):
        # Lấy ảnh ground truth và dự đoán
        y_true_img = y_test[i].astype(np.float32)
        y_pred_img = predictions[i].astype(np.float32)

        # Nếu shape là (H, W, 1), ta squeeze về (H, W)
        if y_true_img.ndim == 3 and y_true_img.shape[2] == 1:
            y_true_img = np.squeeze(y_true_img, axis=2)
        if y_pred_img.ndim == 3 and y_pred_img.shape[2] == 1:
            y_pred_img = np.squeeze(y_pred_img, axis=2)

        # Bây giờ y_true_img và y_pred_img nên có shape (H, W) cho grayscale.
        # Nếu shape khác (ví dụ H, W, C với C>1) thì có thể mở rộng sau này.

        # Flatten để tính MSE/MAE/...
        y_true_flat = y_true_img.flatten()
        y_pred_flat = y_pred_img.flatten()

        # Tính MSE, MAE, RMSE
        mse = np.mean((y_true_flat - y_pred_flat) ** 2)
        mae = np.mean(np.abs(y_true_flat - y_pred_flat))
        rmse = np.sqrt(mse)

        # Tính RMAE = MAE / mean(|y_true|), tránh chia cho 0
        mean_abs_y = np.mean(np.abs(y_true_flat))
        rmae = mae / mean_abs_y if mean_abs_y != 0 else np.nan

        # Tính SSIM cho grayscale
        try:
            # y_true_img và y_pred_img giờ là 2D (H, W)
            ssim_val = ssim(
                y_true_img,
                y_pred_img,
                data_range=data_range
            )
        except ValueError:
            # Nếu SSIM lỗi (ví dụ data_range không phù hợp),
            # tính lại data_range tự động từ dữ liệu thực tế
            dmin = min(float(y_true_img.min()), float(y_pred_img.min()))
            dmax = max(float(y_true_img.max()), float(y_pred_img.max()))
            auto_range = dmax - dmin if (dmax - dmin) > 0 else 1.0
            ssim_val = ssim(
                y_true_img,
                y_pred_img,
                data_range=auto_range
            )

        mse_list.append(mse)
        mae_list.append(mae)
        rmse_list.append(rmse)
        rmae_list.append(rmae)
        ssim_list.append(ssim_val)

    metrics_avg = {
        'MSE': np.mean(mse_list),
        'MAE': np.mean(mae_list),
        'RMSE': np.mean(rmse_list),
        'RMAE': np.nanmean(rmae_list),
        'SSIM': np.mean(ssim_list)
    }

    return metrics_avg

In [ ]:
# Đường dẫn và custom objects như cũ
saved_model_path = '/content/drive/MyDrive/BacSon/EAI_2024/KetQua/CNN_GRU_v2.keras'

# Load generator
generator = load_model(saved_model_path)

# Giả sử X_test_seq có shape (batch_size, 4, 224, 224, 1)
# và y_test có shape (batch_size, 6, 224, 224, 1) – ground truth 6 khung hình tiếp
batch_size = X_test_seq.shape[0]
num_preds = 6  # số khung hình muốn sinh ra

# Khởi tạo window là 4 khung đầu từ X_test_seq
window = X_test_seq.copy()  # shape: (batch_size, 4, H, W, C)

all_preds = []
for t in range(num_preds):
    # Dự đoán 1 bước cho cả batch
    pred = generator.predict(window, batch_size=batch_size)
    # pred shape: (batch_size, H, W, C)

    # Ghép axis frame để window thành (batch_size, 5, H, W, C) tạm
    pred_exp = pred[..., np.newaxis]  # (batch_size, H, W, C, 1) đổi thứ tự nếu cần
    # chuyển về (batch_size, 1, H, W, C)
    pred_frame = np.expand_dims(pred, axis=1)

    all_preds.append(pred_frame)  # lưu lại khung t

    # Cập nhật window: bỏ khung đầu, thêm khung mới
    window = np.concatenate([window[:, 1:], pred_frame], axis=1)
    # giờ window shape lại (batch_size, 4, H, W, C)

# Stack các frame dự đoán thành (batch_size, 6, H, W, C)
predictions = np.concatenate(all_preds, axis=1)

print("Predictions shape:", predictions.shape)
# -> (batch_size, 6, 224, 224, 1)

# Chuyển về uint8 [0,255] nếu cần
predictions_uint8 = (predictions * 255).astype(np.uint8)
y_test_uint8 = (y_test_seq * 255).astype(np.uint8)

# Tính metrics cho từng khung ảnh rồi lưu vào list of dicts
metrics_per_frame = []
for i in range(num_preds):
    m = evaluate_image_metrics(y_test_seq[:, i], predictions[:, i])
    metrics_per_frame.append(m)

# In kết quả cho từng frame
for i, m in enumerate(metrics_per_frame, start=1):
    print(f"--- Frame {i} ---")
    for metric, value in m.items():
        print(f"{metric}: {value:.6f}")
    print()

# Tính trung bình toàn bộ
metrics_avg = {}
for key in metrics_per_frame[0].keys():
    metrics_avg[key] = np.mean([m[key] for m in metrics_per_frame])

print("=== Trung bình cả 6 khung dự đoán ===")
for metric, value in metrics_avg.items():
    print(f"{metric}: {value:.6f}")

In [ ]:
def visualize_results(X_input, y_true, y_pred, num_samples=3):
    batch_size, T_in, H, W, C = X_input.shape
    _, T_out, _, _, _ = y_true.shape

    # Squeeze kênh nếu C=1
    def squeeze_img(img):
        # img shape (..., H, W, C)
        if img.shape[-1] == 1:
            return img[..., 0]
        else:
            return img

    # Chọn indices: nếu batch nhỏ hơn num_samples thì hiển thị hết, else chọn ngẫu nhiên
    if batch_size <= num_samples:
        indices = np.arange(batch_size)
    else:
        indices = np.random.choice(batch_size, size=num_samples, replace=False)

    for idx in indices:
        # Lấy từng mẫu
        Xi = X_input[idx]   # shape (T_in, H, W, C)
        Yi = y_true[idx]    # shape (T_out, H, W, C)
        Pi = y_pred[idx]    # shape (T_out, H, W, C)

        # Squeeze
        Xi = squeeze_img(Xi)  # shape (T_in, H, W) nếu C=1, else (T_in, H, W, C)
        Yi = squeeze_img(Yi)
        Pi = squeeze_img(Pi)

        # Thiết lập figure: 3 hàng, cột tối đa giữa T_in và T_out
        n_cols = max(T_in, T_out)
        figsize = (n_cols * 2, 3 * 2)  # mỗi ảnh ~2 inch
        fig, axes = plt.subplots(3, n_cols, figsize=figsize)

        # Nếu chỉ có 1 hàng hoặc 1 cột, đảm bảo axes là mảng 2D
        if axes.ndim == 1:
            # trường hợp 3 x 1: axes shape (3,), cần reshape thành (3,1)
            axes = axes.reshape(3, 1)

        # Hàng 1: Input frames
        for i in range(n_cols):
            ax = axes[0, i] if n_cols > 1 else axes[0, 0]
            if i < T_in:
                img = Xi[i]
                ax.imshow(img if img.ndim == 2 else None)
                ax.set_title(f"Input {i+1}")
            ax.axis('off')

        # Hàng 2: Ground truth frames
        for i in range(n_cols):
            ax = axes[1, i] if n_cols > 1 else axes[1, 0]
            if i < T_out:
                img = Yi[i]
                ax.imshow(img if img.ndim == 2 else None)
                ax.set_title(f"GT {i+1}")
            ax.axis('off')

        # Hàng 3: Predicted frames
        for i in range(n_cols):
            ax = axes[2, i] if n_cols > 1 else axes[2, 0]
            if i < T_out:
                img = Pi[i]
                ax.imshow(img if img.ndim == 2 else None)
                ax.set_title(f"Pred {i+1}")
            ax.axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
visualize_results(X_test_seq, y_test_seq, predictions, num_samples=6)